# Pandas DataFrame Merge, Join, Concat & Database Operations

This notebook covers essential pandas operations for combining DataFrames:
- **merge**: SQL-style joins (inner, outer, left, right)
- **join**: Index-based joining
- **concat**: Stacking DataFrames vertically/horizontally
- **Database operations**: SQL integration, query execution

Includes enterprise-grade examples and C++ performance comparisons.

## 1. Setup and Sample Data

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta
import time

# Enterprise sample: Customer Orders
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5, 6],
    'name': ['Alice Johnson', 'Bob Smith', 'Charlie Brown', 'Diana Prince', 'Eve Davis', 'Frank Miller'],
    'region': ['North', 'South', 'North', 'East', 'West', 'South'],
    'tier': ['Gold', 'Silver', 'Gold', 'Platinum', 'Silver', 'Gold']
})

orders = pd.DataFrame({
    'order_id': [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    'customer_id': [1, 2, 3, 1, 4, 5, 2, 8],
    'product': ['Laptop', 'Phone', 'Tablet', 'Monitor', 'Keyboard', 'Mouse', 'Printer', 'Scanner'],
    'amount': [1200, 800, 450, 350, 120, 45, 280, 190],
    'order_date': pd.date_range('2024-01-01', periods=8, freq='D')
})

print('=== Customers ===')
print(customers)
print('\n=== Orders ===')
print(orders)

## 2. `merge()` - SQL-style Joins

The `merge()` function is pandas' primary tool for combining DataFrames based on common columns, similar to SQL JOIN operations.

### Syntax:
```python
pd.merge(left, right, how='inner', on=None, left_on=None, right_on=None, suffixes=('_x', '_y'))
```

**Join Types:**
- `inner`: Only matching rows (intersection)
- `outer`: All rows from both (union), fill NaN for missing
- `left`: All rows from left + matching from right
- `right`: All rows from right + matching from left

In [ ]:
# Inner Join - Only matching customer_id
inner_merge = pd.merge(orders, customers, on='customer_id', how='inner')
print('Inner Join:')
print(inner_merge)
print(f'Shape: {inner_merge.shape}')

In [ ]:
# Left Join - All orders, matching customers
left_merge = pd.merge(orders, customers, on='customer_id', how='left')
print('Left Join:')
print(left_merge)
print(f'\nNull customers: {left_merge["name"].isna().sum()}')

In [ ]:
# Outer Join - All records from both DataFrames
outer_merge = pd.merge(orders, customers, on='customer_id', how='outer')
print('Outer Join:')
print(outer_merge)
print(f'\nTotal rows: {len(outer_merge)}')

In [ ]:
# Right Join - All customers, matching orders
right_merge = pd.merge(orders, customers, on='customer_id', how='right')
print('Right Join:')
print(right_merge)

In [ ]:
# Multi-column merge with different column names
employees = pd.DataFrame({
    'emp_id': [1, 2, 3, 4],
    'dept_code': ['HR', 'IT', 'FIN', 'IT'],
    'salary': [75000, 95000, 88000, 92000]
})

departments = pd.DataFrame({
    'dept_id': ['HR', 'IT', 'FIN', 'MKT'],
    'dept_name': ['Human Resources', 'Information Technology', 'Finance', 'Marketing']
})

# Merge on different column names using left_on and right_on
dept_merge = pd.merge(employees, departments, left_on='dept_code', right_on='dept_id', how='left')
print(dept_merge)

### Merge with Validation

In enterprise applications, data validation is critical. Pandas supports merge validation to catch data quality issues.

In [ ]:
# Validate merge relationships
# 'one_to_one' or '1:1': Check if merge keys are unique in both DataFrames
# 'one_to_many' or '1:m': Check if merge keys are unique in left DataFrame
# 'many_to_one' or 'm:1': Check if merge keys are unique in right DataFrame
# 'many_to_many' or 'm:m': No uniqueness check (default)

# This would raise an error if customer_id is not unique in customers
try:
    validated = pd.merge(orders, customers, on='customer_id', validate='many_to_one')
    print('Merge validated successfully (many orders to one customer)')
    print(validated.head())
except Exception as e:
    print(f'Validation error: {e}')

## 3. `join()` - Index-based Joining

The `join()` method is a convenience wrapper around `merge()` that joins on indices by default. It's more efficient when joining on index values.

In [ ]:
# Set index for join operations
customers_idx = customers.set_index('customer_id')
orders_idx = orders.set_index('customer_id')

print('Customers (indexed):')
print(customers_idx)
print('\nOrders (indexed):')
print(orders_idx)

In [ ]:
# Index-based join
joined = orders_idx.join(customers_idx, how='inner')
print('Index-based Inner Join:')
print(joined)

In [ ]:
# Join multiple DataFrames at once
payments = pd.DataFrame({
    'order_id': [1001, 1002, 1003, 1004, 1005],
    'payment_method': ['Credit Card', 'PayPal', 'Credit Card', 'Bank Transfer', 'Credit Card'],
    'payment_status': ['Completed', 'Completed', 'Pending', 'Completed', 'Failed']
}).set_index('order_id')

shipping = pd.DataFrame({
    'order_id': [1001, 1002, 1003, 1004, 1006],
    'shipping_method': ['Express', 'Standard', 'Express', 'Standard', 'Express'],
    'tracking_number': ['TRK001', 'TRK002', 'TRK003', 'TRK004', 'TRK006']
}).set_index('order_id')

# Join multiple DataFrames
full_order_details = orders.set_index('order_id').join([payments, shipping], how='left')
print('Full Order Details (Multiple Joins):')
print(full_order_details)

## 4. `concat()` - Stacking DataFrames

The `concat()` function stacks DataFrames along a particular axis. Unlike `merge`/`join`, it doesn't align on keys—it simply appends.

In [ ]:
# Vertical concatenation (stacking rows)
q1_sales = pd.DataFrame({
    'product': ['Laptop', 'Phone', 'Tablet'],
    'q1_revenue': [150000, 280000, 95000],
    'q1_units': [125, 350, 190]
})

q2_sales = pd.DataFrame({
    'product': ['Laptop', 'Phone', 'Tablet'],
    'q2_revenue': [165000, 310000, 110000],
    'q2_units': [138, 390, 220]
})

# Horizontal concatenation (adding columns)
sales_comparison = pd.concat([q1_sales.set_index('product'), q2_sales.set_index('product')], axis=1)
print('Horizontal Concat (Sales Comparison):')
print(sales_comparison)

In [ ]:
# Vertical concatenation with different schemas
region_north = pd.DataFrame({
    'store': ['Store A', 'Store B'],
    'region': ['North', 'North'],
    'revenue': [50000, 62000]
})

region_south = pd.DataFrame({
    'store': ['Store C', 'Store D', 'Store E'],
    'region': ['South', 'South', 'South'],
    'revenue': [48000, 55000, 71000],
    'manager': ['John', 'Jane', 'Bob']  # Extra column
})

# Stack vertically, NaN for missing values
all_stores = pd.concat([region_north, region_south], ignore_index=True)
print('Vertical Concat (All Stores):')
print(all_stores)

In [ ]:
# Multi-file data loading pattern (common in enterprise ETL)
def load_and_concat(file_pattern, year):
    """Simulate loading multiple monthly files and concatenating"""
    dfs = []
    for month in range(1, 13):
        # Simulate monthly data
        df = pd.DataFrame({
            'date': pd.date_range(f'{year}-{month:02d}-01', periods=5, freq='D'),
            'sales': np.random.randint(1000, 5000, 5),
            'region': np.random.choice(['North', 'South', 'East', 'West'], 5)
        })
        df['month'] = month
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

# Load full year data
yearly_data = load_and_concat('sales_*.csv', 2024)
print(f'Yearly data shape: {yearly_data.shape}')
print(yearly_data.head(10))
print(f'\nMonthly summary:')
print(yearly_data.groupby('month')['sales'].sum())

## 5. Database Operations

Pandas integrates seamlessly with databases via SQLAlchemy and supports direct SQL queries. This is essential for enterprise applications.

In [ ]:
# Create SQLite database for demonstration
conn = sqlite3.connect(':memory:')  # In-memory database
cursor = conn.cursor()

# Create tables
cursor.execute('''
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT,
    unit_price REAL
)
''')

cursor.execute('''
CREATE TABLE order_items (
    item_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,
    FOREIGN KEY (product_id) REFERENCES products(product_id)
)
''')

# Insert sample data
products_data = [
    (1, 'Laptop Pro', 'Electronics', 1299.99),
    (2, 'Wireless Mouse', 'Accessories', 29.99),
    (3, 'USB-C Hub', 'Accessories', 49.99),
    (4, 'Monitor 27"', 'Electronics', 399.99),
    (5, 'Keyboard Mech', 'Accessories', 89.99)
]

order_items_data = [
    (1, 1001, 1, 2),
    (2, 1001, 2, 5),
    (3, 1002, 3, 10),
    (4, 1002, 4, 1),
    (5, 1003, 1, 1),
    (6, 1003, 5, 3)
]

cursor.executemany('INSERT INTO products VALUES (?, ?, ?, ?)', products_data)
cursor.executemany('INSERT INTO order_items VALUES (?, ?, ?, ?)', order_items_data)
conn.commit()

print('Database tables created and populated.')

In [ ]:
# Read SQL query into DataFrame
query = """
SELECT 
    oi.order_id,
    p.product_name,
    p.category,
    oi.quantity,
    p.unit_price,
    (oi.quantity * p.unit_price) as line_total
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
ORDER BY oi.order_id, line_total DESC
"""

order_details = pd.read_sql_query(query, conn)
print('Order Details from SQL:')
print(order_details)
print(f'\nTotal Revenue: ${order_details["line_total"].sum():,.2f}')

In [ ]:
# Write DataFrame to SQL table
new_orders = pd.DataFrame({
    'order_id': [1004, 1004, 1005],
    'product_id': [2, 3, 1],
    'quantity': [20, 15, 1]
})

new_orders.to_sql('order_items_temp', conn, index=False, if_exists='replace')

# Read back to verify
verify = pd.read_sql('SELECT * FROM order_items_temp', conn)
print('Written to SQL:')
print(verify)

In [ ]:
# Complex analytical query
analytics_query = """
SELECT 
    p.category,
    COUNT(DISTINCT oi.order_id) as num_orders,
    SUM(oi.quantity) as total_units,
    SUM(oi.quantity * p.unit_price) as total_revenue,
    AVG(p.unit_price) as avg_unit_price
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC
"""

category_analytics = pd.read_sql_query(analytics_query, conn)
print('Category Analytics:')
print(category_analytics)

In [ ]:
# Parameterized queries (preventing SQL injection)
min_price = 50.0
category_filter = 'Electronics'

param_query = """
SELECT product_name, unit_price 
FROM products 
WHERE unit_price >= ? AND category = ?
"""

filtered = pd.read_sql_query(param_query, conn, params=[min_price, category_filter])
print(f'Products >= ${min_price} in {category_filter}:')
print(filtered)

conn.close()

## 6. Enterprise Example: Sales Analytics Pipeline

A real-world example combining merge, join, concat, and database operations for a complete analytics workflow.

In [ ]:
# Enterprise Sales Analytics Pipeline
np.random.seed(42)

# Master data
products = pd.DataFrame({
    'product_id': range(1, 21),
    'product_name': [f'Product_{i}' for i in range(1, 21)],
    'category': np.random.choice(['Electronics', 'Clothing', 'Food', 'Books'], 20),
    'supplier_id': np.random.randint(100, 105, 20)
})

suppliers = pd.DataFrame({
    'supplier_id': [100, 101, 102, 103, 104],
    'supplier_name': ['TechCorp', 'FashionInc', 'FoodDist', 'BookWorld', 'GlobalSup'],
    'country': ['USA', 'Italy', 'France', 'UK', 'Germany']
})

# Monthly sales data (simulating multiple files)
sales_dfs = []
for month in range(1, 7):
    df = pd.DataFrame({
        'order_date': pd.date_range(f'2024-{month:02d}-01', periods=50, freq='D'),
        'product_id': np.random.randint(1, 21, 50),
        'quantity': np.random.randint(1, 100, 50),
        'region': np.random.choice(['North', 'South', 'East', 'West'], 50)
    })
    sales_dfs.append(df)

# Step 1: Concat all monthly data
all_sales = pd.concat(sales_dfs, ignore_index=True)

# Step 2: Calculate revenue by merging with product prices
product_prices = products[['product_id']].copy()
product_prices['price'] = np.random.uniform(10, 500, 20).round(2)

sales_with_price = pd.merge(all_sales, product_prices, on='product_id')
sales_with_price['revenue'] = sales_with_price['quantity'] * sales_with_price['price']

# Step 3: Enrich with product and supplier info
full_sales = pd.merge(sales_with_price, products, on='product_id')
full_sales = pd.merge(full_sales, suppliers, on='supplier_id')

# Step 4: Analytics
print('=== Top 5 Products by Revenue ===')
top_products = full_sales.groupby('product_name')['revenue'].sum().nlargest(5)
print(top_products)

print('\n=== Revenue by Supplier Country ===')
by_country = full_sales.groupby('country')['revenue'].sum().sort_values(ascending=False)
print(by_country)

print('\n=== Monthly Revenue Trend ===')
monthly = full_sales.groupby(full_sales['order_date'].dt.month)['revenue'].sum()
print(monthly)

## 7. C++ Performance Comparison

Pandas operations are implemented in C/Cython under the hood. Let's compare pandas merge performance with a pure Python approach to illustrate the efficiency gain.

In [ ]:
# Performance Comparison: Pandas vs Pure Python
import time

# Generate large datasets
n = 100000
np.random.seed(42)

df_left = pd.DataFrame({
    'key': np.random.randint(0, n//2, n),
    'value_left': np.random.randn(n)
})

df_right = pd.DataFrame({
    'key': np.random.randint(0, n//2, n//2),
    'value_right': np.random.randn(n//2)
})

# Pandas merge
start = time.perf_counter()
result_pandas = pd.merge(df_left, df_right, on='key', how='inner')
pandas_time = time.perf_counter() - start

# Pure Python equivalent (dict-based lookup)
start = time.perf_counter()
right_dict = {}
for idx, row in df_right.iterrows():
    right_dict.setdefault(row['key'], []).append(row['value_right'])

result_python = []
for idx, row in df_left.iterrows():
    if row['key'] in right_dict:
        for val in right_dict[row['key']]:
            result_python.append((row['key'], row['value_left'], val))
python_time = time.perf_counter() - start

print(f'Pandas merge time: {pandas_time:.4f} seconds')
print(f'Pure Python time:  {python_time:.4f} seconds')
print(f'Speedup: {python_time/pandas_time:.1f}x faster with pandas')
print(f'\nPandas result rows: {len(result_pandas)}')
print(f'Python result rows: {len(result_python)}')

In [ ]:
# Why pandas is faster: C-level optimizations
# pandas uses:
# 1. Hash-based joins (similar to C++ std::unordered_map)
# 2. Vectorized operations via NumPy (C arrays)
# 3. Cython-compiled merge logic

# Concat performance comparison
n_dfs = 100
dfs = [pd.DataFrame({'a': np.random.randn(1000), 'b': np.random.randn(1000)}) for _ in range(n_dfs)]

# Pandas concat
start = time.perf_counter()
result_concat = pd.concat(dfs, ignore_index=True)
concat_time = time.perf_counter() - start

# Pure Python append
start = time.perf_counter()
result_list = []
for df in dfs:
    result_list.extend(df.values.tolist())
list_time = time.perf_counter() - start

print(f'Pandas concat time: {concat_time:.4f} seconds')
print(f'Python list time:   {list_time:.4f} seconds')
print(f'Speedup: {list_time/concat_time:.1f}x faster with pandas')

## 8. Advanced Merge Patterns

### Cross Join (Cartesian Product)
### Merge with Indicator
### Merge Asof (Time-series)

In [ ]:
# Cross Join - Cartesian Product
sizes = pd.DataFrame({'size': ['S', 'M', 'L', 'XL']})
colors = pd.DataFrame({'color': ['Red', 'Blue', 'Green']})

# Create key column for cross join
sizes['_key'] = 1
colors['_key'] = 1

product_variants = pd.merge(sizes, colors, on='_key').drop('_key', axis=1)
print('Product Variants (Cross Join):')
print(product_variants)

In [ ]:
# Merge with indicator - Track where each row came from
df_a = pd.DataFrame({'id': [1, 2, 3, 4], 'val_a': ['a1', 'a2', 'a3', 'a4']})
df_b = pd.DataFrame({'id': [2, 3, 5, 6], 'val_b': ['b2', 'b3', 'b5', 'b6']})

merged = pd.merge(df_a, df_b, on='id', how='outer', indicator=True)
print('Merge with Indicator:')
print(merged)

print('\nRow counts by source:')
print(merged['_merge'].value_counts())

In [ ]:
# Merge Asof - Time-series merge (match on nearest key)
# Useful for aligning time-series data with different frequencies

# Daily prices
prices = pd.DataFrame({
    'date': pd.to_datetime(['2024-01-01', '2024-01-03', '2024-01-05', '2024-01-08']),
    'price': [100.0, 102.5, 98.0, 105.0]
})

# Trade events (irregular timestamps)
trades = pd.DataFrame({
    'date': pd.to_datetime(['2024-01-02 10:00', '2024-01-04 14:30', '2024-01-07 09:15']),
    'trade_size': [500, 300, 700]
})

# Merge asof: match each trade with the most recent price
trades_with_price = pd.merge_asof(trades, prices, on='date')
print('Trades with Nearest Price (merge_asof):')
print(trades_with_price)

## 9. Enterprise Pattern: ETL Pipeline with Merge/Concat

Combining multiple data sources, cleaning, and preparing for analytics.

In [ ]:
# Simulated ETL Pipeline
def extract_crm_data():
    """Extract from CRM system"""
    return pd.DataFrame({
        'customer_id': range(1001, 1021),
        'name': [f'Customer_{i}' for i in range(1, 21)],
        'segment': np.random.choice(['Enterprise', 'SMB', 'Startup'], 20),
        'acquisition_date': pd.date_range('2022-01-01', periods=20, freq='M')
    })

def extract_billing_data(month):
    """Extract from billing system"""
    return pd.DataFrame({
        'customer_id': np.random.randint(1001, 1021, 30),
        'invoice_date': pd.date_range(f'2024-{month:02d}-01', periods=30, freq='D'),
        'amount': np.random.exponential(500, 30).round(2),
        'product': np.random.choice(['Basic', 'Pro', 'Enterprise'], 30)
    })

def transform_data(crm_df, billing_dfs):
    """Transform: merge and aggregate"""
    # Concat all billing data
    all_billing = pd.concat(billing_dfs, ignore_index=True)
    
    # Aggregate billing by customer
    billing_agg = all_billing.groupby('customer_id').agg(
        total_revenue=('amount', 'sum'),
        num_invoices=('amount', 'count'),
        avg_invoice=('amount', 'mean')
    ).reset_index()
    
    # Merge with CRM data
    enriched = pd.merge(crm_df, billing_agg, on='customer_id', how='left')
    
    # Calculate customer lifetime value proxy
    enriched['days_since_acquisition'] = (pd.Timestamp.now() - enriched['acquisition_date']).dt.days
    enriched['ltv_proxy'] = enriched['total_revenue'] / enriched['days_since_acquisition'] * 365
    
    return enriched

# Execute ETL
crm = extract_crm_data()
billing_dfs = [extract_billing_data(m) for m in range(1, 7)]

customer_360 = transform_data(crm, billing_dfs)

print('Customer 360 View:')
print(customer_360[['customer_id', 'name', 'segment', 'total_revenue', 'num_invoices', 'ltv_proxy']].head(10))

print('\n=== Revenue by Segment ===')
print(customer_360.groupby('segment')['total_revenue'].agg(['sum', 'mean', 'count']))

## 10. Summary & Best Practices

| Operation | Use Case | Performance |
|-----------|----------|-------------|
| `merge()` | SQL-style joins on columns | O(n+m) with hash join |
| `join()` | Index-based joining | Faster for index alignment |
| `concat()` | Stacking DataFrames | O(n) linear scan |
| `read_sql()` | Database integration | Depends on DB engine |

**Best Practices:**
1. Use `merge()` for column-based joins; `join()` for index-based
2. Set appropriate `how` parameter (inner/left/outer/right)
3. Use `validate` parameter to catch data quality issues
4. For time-series, consider `merge_asof()` for nearest-match joins
5. Use `indicator=True` during development to debug join behavior
6. Always use parameterized queries for SQL to prevent injection
7. For large datasets, consider chunking with `pd.read_sql()`